In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report,f1_score,roc_auc_score,roc_curve
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder,LabelEncoder,PowerTransformer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTENC

In [2]:
df = pd.read_csv('final_data.csv')
df = df.drop(['education', 'housing', 'campaign_call', 'previous', 'top_jobs',
       'multiple_calls', 'previous_camp_call','contact', 'loan',
       'contacted_before', 'default', 'high_intensity_calls'],axis=1)
df.head()

,age,job,marital,balance,day,month,duration,campaign,pdays,poutcome,y,age_bin,balance_bins,loan_default_risk,duration_min,is_long_cal,pdays_cat,is_q2_calls,week,qtr
0,58,management,married,2143,5,may,261,1,-1,never contacted,no,56-65,high positive,1,less than 5 minutes,0,never contacted,1,week 1,Q2
1,44,technician,single,29,5,may,151,1,-1,never contacted,no,41-55,low positive,1,less than 3 minutes,0,never contacted,1,week 1,Q2
2,33,entrepreneur,married,2,5,may,76,1,-1,never contacted,no,31-40,low positive,1,less than 2 minutes,0,never contacted,1,week 1,Q2
3,47,blue-collar,married,1506,5,may,92,1,-1,never contacted,no,41-55,high positive,1,less than 2 minutes,0,never contacted,1,week 1,Q2
4,33,blue-collar,single,1,5,may,198,1,-1,never contacted,no,31-40,low positive,0,less than 4 minutes,0,never contacted,1,week 1,Q2


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44444 entries, 0 to 44443
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   age                44444 non-null  int64 
 1   job                44444 non-null  object
 2   marital            44444 non-null  object
 3   balance            44444 non-null  int64 
 4   day                44444 non-null  int64 
 5   month              44444 non-null  object
 6   duration           44444 non-null  int64 
 7   campaign           44444 non-null  int64 
 8   pdays              44444 non-null  int64 
 9   poutcome           44444 non-null  object
 10  y                  44444 non-null  object
 11  age_bin            44444 non-null  object
 12  balance_bins       44444 non-null  object
 13  loan_default_risk  44444 non-null  int64 
 14  duration_min       44444 non-null  object
 15  is_long_cal        44444 non-null  int64 
 16  pdays_cat          44444 non-null  objec

In [4]:
x = df.drop(['y'],axis=1)
y = df['y']
y = LabelEncoder().fit_transform(y)

In [5]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)

In [6]:
# Cardianl = job,marital,day,month,poutcome,loan_default_risk ,is_long_cal ,week ,qtr ,is_q2_calls
# Ordinal = campaign , age_bin ,balance_bin , duration_min ,pdays_cat    
# numerical = age ,balance , duration (needs scalling)

## (a) Basemodel

In [7]:
models_dict = {
    'LogisticRegression':LogisticRegression(random_state=0),
    'SVC':SVC(random_state=0,probability=True),
    'RandomForestClassifier':RandomForestClassifier(random_state=0),
    'GradientBoostingClassifier':GradientBoostingClassifier(random_state=0),
    'GaussianNB':GaussianNB(),
    'XGBClassifier':XGBClassifier(),
    'LGBMClassifier':LGBMClassifier(random_state=0),
    'CatBoostClassifier':CatBoostClassifier(random_seed=0)
}


def score(model_name,model):
    
    output =[]
    output.append(model_name)

    power_pipe = Pipeline([
    ('log_transformer',PowerTransformer()),
    ('scalling',StandardScaler())
    ])

    preprocessor = ColumnTransformer([
            ('log_pipe',power_pipe,['age','balance']),

            ('cardinal',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),
             ['job','marital','day','month','poutcome','loan_default_risk' ,'is_long_cal' ,'week' ,'qtr' ,'is_q2_calls']),

            ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),
              ['campaign','age_bin','balance_bins','duration_min','pdays_cat']),
              
            ('numerical',StandardScaler(),['duration','pdays'])
                                      ])

    pipe = Pipeline([
    ('preprocessor',preprocessor),
    ('model',model)
    ])

    pipe.fit(x_train,y_train)
    y_pred_roc = pipe.predict_proba(x_test)[:,1]
    y_pred = pipe.predict(x_test)

    output.append(roc_auc_score(y_test,y_pred_roc))
    output.append(f1_score(y_test,y_pred))

    return output

model_output = []
for model_name,model in models_dict.items():
    model_output.append(score(model_name,model))

models = pd.DataFrame(model_output,columns=['Model','ROC_AUC','F1_Score']).sort_values(by='ROC_AUC',ascending=False)
models

[LightGBM] [Info] Number of positive: 4248, number of negative: 31307
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1052
[LightGBM] [Info] Number of data points in the train set: 35555, number of used features: 84
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.119477 -> initscore=-1.997393
[LightGBM] [Info] Start training from score -1.997393


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Learning rate set to 0.047333
0:	learn: 0.6303761	total: 140ms	remaining: 2m 19s
1:	learn: 0.5816813	total: 146ms	remaining: 1m 12s
2:	learn: 0.5405436	total: 152ms	remaining: 50.4s
3:	learn: 0.5098554	total: 157ms	remaining: 39.1s
4:	learn: 0.4799182	total: 163ms	remaining: 32.4s
5:	learn: 0.4484456	total: 168ms	remaining: 27.8s
6:	learn: 0.4206941	total: 174ms	remaining: 24.6s
7:	learn: 0.4001398	total: 179ms	remaining: 22.2s
8:	learn: 0.3782417	total: 185ms	remaining: 20.4s
9:	learn: 0.3636119	total: 192ms	remaining: 19s
10:	learn: 0.3493028	total: 198ms	remaining: 17.8s
11:	learn: 0.3378317	total: 203ms	remaining: 16.8s
12:	learn: 0.3265710	total: 210ms	remaining: 15.9s
13:	learn: 0.3167861	total: 215ms	remaining: 15.2s
14:	learn: 0.3088347	total: 221ms	remaining: 14.5s
15:	learn: 0.3023763	total: 227ms	remaining: 14s
16:	learn: 0.2952131	total: 232ms	remaining: 13.4s
17:	learn: 0.2916172	total: 239ms	remaining: 13s
18:	learn: 0.2867068	total: 244ms	remaining: 12.6s
19:	learn: 0.28

,Model,ROC_AUC,F1_Score
7,CatBoostClassifier,0.932035,0.549499
6,LGBMClassifier,0.928922,0.537297
5,XGBClassifier,0.924556,0.527655
2,RandomForestClassifier,0.923449,0.497379
3,GradientBoostingClassifier,0.919805,0.497915
0,LogisticRegression,0.905143,0.453066
1,SVC,0.900160,0.421823
4,GaussianNB,0.841191,0.461187


# Using Class_weight = Balanced

In [8]:
scale_pos_weight = (y_train[y_train == 0].shape[0]/y_train[y_train == 1].shape[0]) # For XGBoost

models_dict = {
    'LogisticRegression':LogisticRegression(random_state=0,class_weight='balanced',n_jobs=-1),
    'SVC':SVC(random_state=0,probability=True,class_weight='balanced'),
    'DecisionTreeClassifier':DecisionTreeClassifier(random_state=0,class_weight='balanced'),
    'RandomForestClassifier':RandomForestClassifier(random_state=0,class_weight='balanced',n_jobs=-1),
    'GradientBoostingClassifier':GradientBoostingClassifier(random_state=0),
    'GaussianNB':GaussianNB(),
    'XGBClassifier':XGBClassifier(random_state=0,scale_pos_weight=scale_pos_weight),
    'LGBMClassifier':LGBMClassifier(random_state=0,class_weight='balanced',n_jobs=-1),
    'CatBoostClassifier':CatBoostClassifier(random_state=0,auto_class_weights='Balanced',verbose=False)
}


def score(model_name,model):
    
    output =[]
    output.append(model_name)

    power_pipe = Pipeline([
    ('log_transformer',PowerTransformer()),
    ('scalling',StandardScaler())
    ])

    preprocessor = ColumnTransformer([
            ('log_pipe',power_pipe,['age','balance']),

            ('cardinal',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),
             ['job','marital','day','month','poutcome','loan_default_risk' ,'is_long_cal' ,'week' ,'qtr' ,'is_q2_calls']),

            ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),
              ['campaign','age_bin','balance_bins','duration_min','pdays_cat']),
              
            ('numerical',StandardScaler(),['duration','pdays'])
                                      ])

    pipe = Pipeline([
    ('preprocessor',preprocessor),
    ('model',model)
    ])

    pipe.fit(x_train,y_train)
    y_pred_roc = pipe.predict_proba(x_test)[:,1]
    y_pred = pipe.predict(x_test)

    output.append(roc_auc_score(y_test,y_pred_roc))
    output.append(f1_score(y_test,y_pred))

    return output

model_output = []
for model_name,model in models_dict.items():
    model_output.append(score(model_name,model))

models = pd.DataFrame(model_output,columns=['Model','ROC_AUC','F1_Score']).sort_values(by='ROC_AUC',ascending=False)
models

[LightGBM] [Info] Number of positive: 4248, number of negative: 31307
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1052
[LightGBM] [Info] Number of data points in the train set: 35555, number of used features: 84
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,ROC_AUC,F1_Score
8,CatBoostClassifier,0.930446,0.589674
7,LGBMClassifier,0.928434,0.568003
3,RandomForestClassifier,0.923838,0.484666
1,SVC,0.922920,0.552679
6,XGBClassifier,0.922447,0.592540
4,GradientBoostingClassifier,0.919805,0.497915
0,LogisticRegression,0.907167,0.532508
5,GaussianNB,0.841191,0.461187
2,DecisionTreeClassifier,0.696313,0.456026
